In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [12]:
import torch

In [16]:
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")


prime_utterances = [
    "Who are you and what do you do?",
    "I am a helpful, factual, and polite AI assistant. I am here to provide accurate explanations to your questions without any sarcasm."
]

chat_history_ids = None

for utterance in prime_utterances:
    new_ids = tokenizer.encode(utterance + tokenizer.eos_token, return_tensors='pt')
    if chat_history_ids is not None:
        chat_history_ids = torch.cat([chat_history_ids, new_ids], dim=-1)
    else:
        chat_history_ids = new_ids


while True:
    user_input = input("You: ")
    
    # exit condition
    if user_input.lower() in ["exit", "quit"]:
        break

    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)

    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50, 
        top_p=0.9,
        temperature=0.6  # Lowered from 0.8 to keep it more grounded and factual
    )

    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    
    print(f"Chatbot: {response}")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Chatbot: Hello! I am your AI assistant. How can I help you today?


You:  Hello


Chatbot: Hello! I'm a bot, I respond to all questions.


You:  exit
